In [0]:
from pyspark.sql import functions as F
silver_employees = spark.read.table('silver_employees')


In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

gold_department_metrics = (
    silver_employees
    .groupBy('department')
    .agg(
        F.count(F.col('emp_id')).alias('total_employees'),
        F.round(F.avg(F.col('salary')), 2).alias('avg_salary'),
        F.round(F.avg(F.col('pto_days_used')), 2).alias('avg_pto_used')
    )
)
display(gold_department_metrics)


department,count(emp_id AS total_employees),"round(avg(salary) AS avg_salary, 2)","round(avg(pto_days_used) AS avg_pto_used, 2)"
Engineering,15,94500.0,10.2
Finance,7,74714.29,5.71
Management,5,110200.0,18.8
HR,6,83200.0,8.83
Marketing,7,81666.67,6.71
Sales,8,81250.0,5.38


In [0]:
window_spec = (
    Window
    .partitionBy('department')
    .orderBy(F.col('pto_days_used').desc())
)
gold_top_pto_employees = (
    silver_employees
    .withColumn('rank', F.row_number().over(window_spec))
    .filter(F.col('rank') == 1) # Her departmanın 1. sıradakini alma
    .drop('rank') # İsteğe bağlı: rank kolonunu kaldırıp temiz tablo bırakma
)
display(gold_top_pto_employees)

department,emp_id,first_name,hire_date,last_name,pto_days_allowed,pto_days_used,salary,sick_days_allowed,sick_days_used,ingestion_timestamp,email,remain_pto,remain_sick,days_since_hire
Engineering,1041,Edward,2017-09-14,Perez,25,21,112000,12,5,2026-10-07T16:03:23.221Z,edward.perez@gmail.com,4,7,3310
Finance,1044,Laura,2020-01-08,Phillips,18,13,89000,10,4,2026-10-07T16:03:23.221Z,laura.phillips@gmail.com,5,6,2464
HR,1013,Thomas,2021-12-01,White,15,15,83000,10,3,2026-10-07T16:03:23.221Z,thomas.white@gmail.com,0,7,1771
Management,1029,Steven,2016-11-10,King,25,24,125000,12,8,2026-10-07T16:03:23.221Z,steven.king@gmail.com,1,4,3618
Marketing,1009,Jennifer,2022-10-05,Taylor,15,14,null,8,6,2026-10-07T16:03:23.221Z,jennifer.taylor@gmail.com,1,2,1463
Sales,1026,Sandra,2019-08-14,Allen,20,10,93000,10,2,2026-10-07T16:03:23.221Z,sandra.allen@gmail.com,10,8,2611
